# Análisis exploratorio: Colombianos detenidos en el exterior

| Campo | Valor |
|-------|-------|
| Fuente | Ministerio de Relaciones Exteriores (Cancillería), vía datos.gov.co |
| URL | https://www.datos.gov.co/Estad-sticas-Nacionales/Colombianos-detenidos-en-el-exterior/e97j-vuf7/about_data |
| Fecha de descarga | 28 de septiembre de 2026 |
| Tamaño | 388.148 filas x 12 columnas |
| Cobertura | 17/09/2018 a 01/11/2025, 64 fechas de publicación |
| Granularidad | (la completamos abajo, al entenderla) |

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [15]:
# low_memory=False evita el DtypeWarning: el archivo es grande y pandas
# adivinaría tipos por pedazos. Así lo lee completo antes de decidir.
df = pd.read_csv('Colombianos_detenidos_en_el_exterior_20260928.csv', low_memory=False)
df_original = df.copy()   # copia intacta para comparar "antes y después" al final

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 388148
Columnas: 12


In [16]:
fechas = pd.to_datetime(df['FECHA PUBLICACIÓN'])
print("Primera publicación:", fechas.min().date())
print("Última publicación: ", fechas.max().date())
print("Fechas distintas:   ", fechas.nunique())

por_fecha = df.groupby('FECHA PUBLICACIÓN')['CANTIDAD'].agg(personas='sum', filas='size')
print()
print(por_fecha.head(3))
print(por_fecha.tail(3))
print()
print("Suma de CANTIDAD en TODAS las fechas:", f"{df['CANTIDAD'].sum():,}")
print("Suma de CANTIDAD en la última fecha: ", f"{por_fecha['personas'].iloc[-1]:,}")

Primera publicación: 2018-09-17
Última publicación:  2025-11-01
Fechas distintas:    64

                   personas  filas
FECHA PUBLICACIÓN                 
2018-09-17            23284   4845
2019-01-01            24232   4997
2019-07-04            25307   5206
                   personas  filas
FECHA PUBLICACIÓN                 
2025-09-01            27911   6124
2025-10-01            27804   6115
2025-11-01            27586   6117

Suma de CANTIDAD en TODAS las fechas: 1,862,305
Suma de CANTIDAD en la última fecha:  27,586


1. Justificación de Granularidad y Carga
Justificación de Granularidad:
Cada fila del dataset no representa a una persona individual, sino un grupo consolidado de personas (CANTIDAD) que comparten las mismas características categóricas dentro de una fecha de publicación específica. El dataset contiene 64 "fotos" o reportes periódicos entre 2018 y 2025.

Decisión: Se determina que la columna CANTIDAD no debe sumarse de forma global a través de todas las fechas de publicación, ya que esto incurriría en un conteo múltiple de las mismas personas a lo largo del tiempo. Para análisis puntuales de estado actual se tomará la última foto disponible (27.586 personas), mientras que para análisis de evolución se comparará el total de cada foto por separado.

In [17]:
df.info()
df.head(3).T   # .T gira la tabla: cada columna queda como fila y se lee mejor

<class 'pandas.DataFrame'>
RangeIndex: 388148 entries, 0 to 388147
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   FECHA PUBLICACIÓN           388148 non-null  str  
 1   PAIS PRISIÓN                388148 non-null  str  
 2   CONSULADO                   388148 non-null  str  
 3   DELITO                      388148 non-null  str  
 4   EXTRADITADO Y O REPATRIADO  388148 non-null  str  
 5   SITUACIÓN JURÍDICA          388148 non-null  str  
 6   GÉNERO                      388148 non-null  str  
 7   GRUPO EDAD                  388148 non-null  str  
 8   UBICACIÓN PAÍS              121827 non-null  str  
 9   CANTIDAD                    388148 non-null  int64
 10  LATITUD                     335420 non-null  str  
 11  LONGITUD                    335420 non-null  str  
dtypes: int64(1), str(11)
memory usage: 76.5 MB


,0,1,2
FECHA PUBLICACIÓN,2018-09-17,2018-09-17,2018-09-17
PAIS PRISIÓN,ECUADOR,ITALIA,ESTADOS UNIDOS
CONSULADO,C. QUITO,C. ROMA,C. NUEVA YORK
DELITO,NARCOTRÁFICO,OTROS,EXTORSIÓN
EXTRADITADO Y O REPATRIADO,DESCONOCIDO,DESCONOCIDO,DESCONOCIDO
SITUACIÓN JURÍDICA,EN JUICIO,EN INVESTIGACIÓN,CONDENADO
GÉNERO,MASCULINO,MASCULINO,MASCULINO
GRUPO EDAD,DESCONOCIDO,DESCONOCIDO,ADULTO
UBICACIÓN PAÍS,"(-1.831239, -78.183406)","(41.87194, 12.56738)","(37.09024, -95.712891)"
CANTIDAD,45,1,1


In [18]:
# Nulos que pandas sí ve
nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(1)
print(pd.DataFrame({'nulos': nulos, '%': pct})[nulos > 0])

# Duplicados exactos
print("\nDuplicados:", df.duplicated().sum())

# Nulos disfrazados: pandas NO los cuenta, pero lo son
print("\n'DESCONOCIDO' por columna:")
for col in df.select_dtypes(exclude='number').columns:
    k = (df[col].astype(str).str.strip().str.upper() == 'DESCONOCIDO').sum()
    if k > 0:
        print(f"  {col}: {k:,} ({k/len(df)*100:.1f}%)")

                 nulos     %
UBICACIÓN PAÍS  266321 68.60
LATITUD          52728 13.60
LONGITUD         52728 13.60

Duplicados: 32598

'DESCONOCIDO' por columna:
  PAIS PRISIÓN: 230,170 (59.3%)
  DELITO: 38,984 (10.0%)
  EXTRADITADO Y O REPATRIADO: 365,621 (94.2%)
  GÉNERO: 267 (0.1%)
  GRUPO EDAD: 115,356 (29.7%)


In [19]:
for col in ['EXTRADITADO Y O REPATRIADO', 'SITUACIÓN JURÍDICA', 'GÉNERO', 'GRUPO EDAD']:
    print(f"\n{col}")
    print(df[col].value_counts())


EXTRADITADO Y O REPATRIADO
EXTRADITADO Y O REPATRIADO
DESCONOCIDO                  365621
EXTRADICION                   21833
REPATRIACION                    585
EXTRADICION, REPATRIACION       109
Name: count, dtype: int64

SITUACIÓN JURÍDICA
SITUACIÓN JURÍDICA
CONDENADO                                163103
EN INVESTIGACI�N                          97306
EN JUICIO                                 55427
EN INVESTIGACIÓN                          44378
NO REPORTA - CONFIDENCIALIDAD ESTATAL     12403
EN ESPERA DE DEPORTACI�N                  10171
EN ESPERA DE DEPORTACIÓN                   4047
EXTRADITADO                                1313
Name: count, dtype: int64

GÉNERO
GÉNERO
MASCULINO      305646
FEMENINO        81804
OTRO              410
DESCONOCIDO       267
NO_BINARIO         21
Name: count, dtype: int64

GRUPO EDAD
GRUPO EDAD
ADULTO              172852
DESCONOCIDO         115356
ADULTO JOVEN         50805
ADULTO MAYOR         45522
PRIMERA INFANCIA      1548
ADOLESCENTE      

In [20]:
df['CANTIDAD'].describe()

count   388,148.00
mean          4.80
std          16.87
min           1.00
25%           1.00
50%           1.00
75%           3.00
max         460.00
Name: CANTIDAD, dtype: float64

In [21]:
# '\ufffd' es el carácter de reemplazo: aparece cuando el archivo original perdió una tilde o una Ñ
for col in df.select_dtypes(exclude='number').columns:
    malos = df[col].str.contains('\ufffd', na=False)
    if malos.any():
        print(f"\n{col}: {malos.sum():,} filas afectadas")
        print(df.loc[malos, col].value_counts())


PAIS PRISIÓN: 8,216 filas afectadas
PAIS PRISIÓN
ESPA�A     7930
CURA�AO     286
Name: count, dtype: int64

CONSULADO: 712 filas afectadas
CONSULADO
C. PUERTO ESPA�A    712
Name: count, dtype: int64

DELITO: 79,877 filas afectadas
DELITO
NARCOTR�FICO                             61954
NO REPORTA � CONFIDENCIALIDAD ESTATAL     8276
EXTORSI�N                                 4351
CONDUCCI�N TEMERARIA                      1624
DA�OS                                     1501
FALSIFICACI�N                             1341
DELITOS POL�TICOS                          381
COACCI�N                                   344
CELEBRACI�N INDEBIDA DE CONTRATOS          105
Name: count, dtype: int64

SITUACIÓN JURÍDICA: 107,477 filas afectadas
SITUACIÓN JURÍDICA
EN INVESTIGACI�N            97306
EN ESPERA DE DEPORTACI�N    10171
Name: count, dtype: int64


In [22]:
arreglos = {
    'ESPA�A': 'ESPAÑA', 'CURA�AO': 'CURAÇAO', 'C. PUERTO ESPA�A': 'C. PUERTO ESPAÑA',
    'NARCOTR�FICO': 'NARCOTRÁFICO', 'EXTORSI�N': 'EXTORSIÓN',
    'NO REPORTA � CONFIDENCIALIDAD ESTATAL': 'NO REPORTA - CONFIDENCIALIDAD ESTATAL',
    'CONDUCCI�N TEMERARIA': 'CONDUCCIÓN TEMERARIA', 'DA�OS': 'DAÑOS',
    'FALSIFICACI�N': 'FALSIFICACIÓN', 'DELITOS POL�TICOS': 'DELITOS POLÍTICOS',
    'COACCI�N': 'COACCIÓN', 'CELEBRACI�N INDEBIDA DE CONTRATOS': 'CELEBRACIÓN INDEBIDA DE CONTRATOS',
    'EN INVESTIGACI�N': 'EN INVESTIGACIÓN', 'EN ESPERA DE DEPORTACI�N': 'EN ESPERA DE DEPORTACIÓN',
}
cols_texto = ['PAIS PRISIÓN', 'CONSULADO', 'DELITO', 'SITUACIÓN JURÍDICA']

antes = {c: df[c].nunique() for c in cols_texto}
df[cols_texto] = df[cols_texto].replace(arreglos)

for c in cols_texto:
    print(f"{c}: {antes[c]} -> {df[c].nunique()} categorías")

PAIS PRISIÓN: 93 -> 91 categorías
CONSULADO: 121 -> 120 categorías
DELITO: 40 -> 32 categorías
SITUACIÓN JURÍDICA: 8 -> 6 categorías


In [23]:
# Fecha: de texto a fecha real
df['FECHA PUBLICACIÓN'] = pd.to_datetime(df['FECHA PUBLICACIÓN'])

# Latitud y longitud: vienen como texto con coma decimal ("-1,831239")
for c in ['LATITUD', 'LONGITUD']:
    nulos_antes = df[c].isna().sum()
    df[c] = pd.to_numeric(df[c].str.replace(',', '.'), errors='coerce')
    print(f"{c}: nulos antes {nulos_antes:,} | después {df[c].isna().sum():,}")

print(df[['FECHA PUBLICACIÓN', 'LATITUD', 'LONGITUD']].dtypes)

LATITUD: nulos antes 52,728 | después 52,728
LONGITUD: nulos antes 52,728 | después 52,728
FECHA PUBLICACIÓN    datetime64[us]
LATITUD                     float64
LONGITUD                    float64
dtype: object


2. Justificación de Texto Dañado y Tipos de Datos
Justificación de Corrección de Texto y Conversión de Tipos:
Caracteres corruptos ($\text{\ufffd}$): Se identificó pérdida de tildes y la letra Ñ por inconsistencias de codificación de origen en 4 columnas. Se optó por una corrección mediante diccionario explícito en lugar de eliminar filas, rescatando más de 100.000 registros e integrando categorías duplicadas (por ejemplo: NARCOTRFICO a NARCOTRÁFICO y EN INVESTIGACIN a EN INVESTIGACIÓN).
Conversión de tipos: FECHA PUBLICACIÓN se transformó a datetime64 para posibilitar agrupaciones temporales. Las coordenadas (LATITUD y LONGITUD) venían en formato de texto con coma decimal; se convirtieron a float64 previa sustitución de comas por puntos. Se verificó con errors='coerce' que la cantidad de nulos permaneció exactamente igual (52.728), garantizando que no se destruyó información válida.

In [24]:
dup = df[df.duplicated()]
print(dup.groupby('FECHA PUBLICACIÓN').size().sort_values(ascending=False))

FECHA PUBLICACIÓN
2022-01-24    27005
2023-01-04     5591
2025-10-01        1
2025-11-01        1
dtype: int64


In [25]:
cargas_repetidas = pd.to_datetime(['2022-01-24', '2023-01-04'])
en_cargas = df['FECHA PUBLICACIÓN'].isin(cargas_repetidas)

filas_antes = len(df)
df = df[~(df.duplicated() & en_cargas)]   # borra duplicados SOLO en esas 2 fechas
print(f"Filas: {filas_antes:,} -> {len(df):,}")

por_fecha = df.groupby('FECHA PUBLICACIÓN')['CANTIDAD'].agg(personas='sum', filas='size')
print(por_fecha.loc[cargas_repetidas])
print(por_fecha['personas'].describe())

Filas: 388,148 -> 355,552
            personas  filas
2022-01-24     26869   5401
2023-01-04     27918   5591
count       64.00
mean    26,563.16
std        797.29
min     23,284.00
25%     26,262.75
50%     26,483.00
75%     26,940.50
max     28,058.00
Name: personas, dtype: float64


3. Justificación de Tratamiento de Duplicados 
Justificación de Eliminación de Duplicados:
Se detectaron 32.598 filas duplicadas exactas. Al analizar su distribución temporal, se halló que el 99.99% de ellos pertenecían a dos fechas específicas:

24/01/2022: 27.005 duplicados (la foto se cargó 6 veces consecutivas de forma idéntica).

04/01/2023: 5.591 duplicados (la foto se cargó 2 veces).

Decisión: Se filtraron los duplicados pertenecientes únicamente a estas dos fechas de carga errónea, reduciendo el dataset de 388.148 a 355.552 filas. Esta limpieza estabilizó el volumen por foto en el rango esperado (entre 23.284 y 28.058 personas por publicación) sin alterar la representatividad de las demás fechas.

In [26]:
anio = df['FECHA PUBLICACIÓN'].dt.year
total = df.groupby(anio)['CANTIDAD'].sum()
desc = df[df['PAIS PRISIÓN'] == 'DESCONOCIDO'].groupby(anio)['CANTIDAD'].sum()
print((desc / total * 100).round(1))   # % de personas con país desconocido, por año

FECHA PUBLICACIÓN
2018    9.30
2019   12.60
2020   15.60
2021   17.20
2022   28.30
2023   95.00
2024   95.60
2025   95.30
Name: CANTIDAD, dtype: float64


4. Hallazgo sobre Registro de País y Ubicación 
Hallazgo sobre Incompletitud Estructural de Variables:
Al evaluar los nulos en PAIS PRISIÓN, se observó un cambio drástico en la metodología de reporte estatal:

Entre 2018 y 2022 el país de prisión se reportaba adecuadamente (menos del 28% DESCONOCIDO).

A partir de 2023, más del 95% de los registros etiquetan el país como DESCONOCIDO.

Decisión: No se imputarán ni eliminarán estos registros para no distorsionar el volumen total. Para análisis del país de estancia actual se utilizará la columna CONSULADO (completitud del 100%), mientras que para análisis específicos por país de prisión se acotará el rango temporal al periodo 2018–2022.